In [10]:
# correct working directory only once 
if not "working_directory_corrected" in vars():
    %cd ..
    working_directory_corrected = True

# Data generation

The cell below simply generates a data set, the size of which is adaptable by changing "num_samples".

Have a look in "utils" to see how the function works.

In [28]:
from utils.generate_jepa_heuristic_dataset import generate_jepa_heuristic_dataset

heuristic_dataset = generate_jepa_heuristic_dataset(num_samples=100000)


Initializing Environment and Model...
18:04:09 | INFO  | utils.py    | Loading checkpoint from folder /Users/richard/.cache/huggingface/hub/models--quentinll--lewm-tworooms/snapshots/77adaae0bc31deab21c93740d1f8bb947cd0bdec...
18:04:09 | INFO  | utils.py    | Created ViT-tiny from scratch with config: {'hidden_size': 192, 'num_hidden_layers': 12, 'num_attention_heads': 3, 'intermediate_size': 768, 'image_size': 224, 'patch_size': 14}
Generating 100000 samples...
Progress: 100/100000
Progress: 200/100000
Progress: 300/100000
Progress: 400/100000
Progress: 500/100000
Progress: 600/100000
Progress: 700/100000
Progress: 800/100000
Progress: 900/100000
Progress: 1000/100000
Progress: 1100/100000
Progress: 1200/100000
Progress: 1300/100000
Progress: 1400/100000
Progress: 1500/100000
Progress: 1600/100000
Progress: 1700/100000
Progress: 1800/100000
Progress: 1900/100000
Progress: 2000/100000
Progress: 2100/100000
Progress: 2200/100000
Progress: 2300/100000
Progress: 2400/100000
Progress: 2500

In [29]:
# Saving the dataset to a pickle file, so we don't have to regenerate it for every restart of the notebook kernel
heuristic_dataset.to_pickle("data/heuristic_dataset.pkl") # PLEASE ADAPT THE NAME OF YOUR DATASET SO YOU DON'T OVERWRITE THE OTHERS

# heuristic_dataset = pd.read_pickle("data/heuristic_dataset.pkl")
# this returns the pickle to a dataframe

In [30]:
from sklearn.model_selection import train_test_split
import numpy as np

agent_encs = np.array(heuristic_dataset['agent_pos_enc'].tolist(), dtype=float)
goal_encs = np.array(heuristic_dataset['goal_pos_enc'].tolist(), dtype=float)

X = np.hstack([agent_encs, goal_encs])
# I also tried using goal_encs-agent_encs which (over 1000 samples) ended in atrocious results. Instead I went for a concatenation of the absolute position of both, letting the machine learning model decide how to utilize them.
y = heuristic_dataset['heuristic'].values.astype(float)

# Split the available data
x_train, x_test, y_train, y_test = train_test_split(X, y, train_size=0.8, test_size=0.2)

array([138.52556774, 130.94057914, 139.65746442, ..., 194.1871395 ,
       117.93319994, 194.52729473], shape=(20000,))

In [31]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import pearsonr, spearmanr

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(n_estimators=100, random_state=42),
    "MLP Neural Net": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=1000, random_state=42)
}

results = {}

for name, model in models.items():
    # Train on Training set
    model.fit(x_train, y_train)

    # Predict on Test set
    y_pred = model.predict(x_test)

    # Calculate Metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    pearson, _ = pearsonr(y_test, y_pred)
    spearman, _ = spearmanr(y_test, y_pred)

    results[name] = {
        "MAE": mae,
        "MSE": mse,
        "R2": r2,
        "Pearson": pearson,
        "Spearman": spearman
    }

# Convert to DataFrame for a clean comparison table
import pandas as pd
results_df = pd.DataFrame(results).T
print(results_df)


'''
results for 1000 samples:

                          MAE           MSE        R2   Pearson  Spearman
 Linear Regression  80.052369  16621.367152 -3.060178  0.168652  0.280871
 Random Forest      47.974281   3404.963886  0.168254  0.419066  0.396016
 MLP Neural Net     51.462562   3917.863676  0.042965  0.269675  0.256868


results for 100,000 samples:

                           MAE          MSE        R2   Pearson  Spearman
  Linear Regression  44.741615  3003.998806  0.198518  0.446034  0.428527
  Random Forest      36.403585  2032.062045  0.457836  0.682042  0.641928
  MLP Neural Net     45.797940  3115.107466  0.168874  0.428174  0.402352

'''

                         MAE          MSE        R2   Pearson  Spearman
Linear Regression  44.741615  3003.998806  0.198518  0.446034  0.428527
Random Forest      36.403585  2032.062045  0.457836  0.682042  0.641928
MLP Neural Net     45.797940  3115.107466  0.168874  0.428174  0.402352
